# Combined Models Tradeoff Analysis

This notebook merges **EfficientNetV2**, **ViT**, and **ResNet18** strategy metrics into one table and plots:
- Performance vs wall-clock training time
- Performance vs inference speed

Outputs are written to `Combined_Models/`:
- `combined_strategies_tradeoff.csv`
- `combined_best_models_tradeoff.csv`
- `combined_tradeoff_wallclock.png`
- `combined_tradeoff_inference.png`


In [16]:
from __future__ import annotations

import json
import math
import re
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Tuple

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd


@dataclass(frozen=True)
class StrategyRecord:
    model_family: str
    strategy: str
    clean_test_acc: Optional[float] = None
    clean_test_loss: Optional[float] = None
    best_val_acc: Optional[float] = None
    train_seconds: Optional[float] = None
    inference_ips: Optional[float] = None
    trainable_params: Optional[float] = None
    source_clean: str = ""
    source_speed: str = ""
    source_train: str = ""


STRATEGY_ALIASES = {
    "linear_probe": "linear_probe",
    "batchnorm_tuning": "batchnorm_tuning",
    "adapter_tuning": "adapter_tuning",
    "lora_tuning": "lora_tuning",
    "full_finetune": "full_finetune",
    "Linear Probing": "linear_probe",
    "BatchNorm Tuning": "batchnorm_tuning",
    "Adapter Tuning": "adapter_tuning",
    "LoRA": "lora_tuning",
    "Full Fine-tuning": "full_finetune",
}

RESNET18_INFERENCE_IPS = {
    "linear_probe": 327.568611,
    "batchnorm_tuning": 300.089071,
    "adapter_tuning": 343.449834,
    "lora_tuning": 329.622109,
    "full_finetune": 333.118427,
    "custom_model": 333.118427,
}

EFFICIENTNETV2_INFERENCE_IPS = {
    "linear_probe": 403.022816,
    "batchnorm_tuning": 401.489927,
    "adapter_tuning": 404.531037,
    "lora_tuning": 349.559394,
    "full_finetune": 407.766329,
}


def as_canonical_strategy(name: str) -> str:
    if name in STRATEGY_ALIASES:
        return STRATEGY_ALIASES[name]
    return name.strip().lower().replace(" ", "_").replace("-", "_")


def clean_float(x) -> Optional[float]:
    if x is None:
        return None
    if isinstance(x, str):
        s = x.strip()
        if s == "" or s.lower() == "nan":
            return None
        try:
            return float(s)
        except ValueError:
            return None
    try:
        val = float(x)
    except Exception:
        return None
    if math.isnan(val):
        return None
    return val


def load_json(path: Path) -> dict:
    return json.loads(path.read_text())


def load_history_records(
    model_family: str,
    history_map: Dict[str, Path],
) -> Dict[str, StrategyRecord]:
    out: Dict[str, StrategyRecord] = {}
    for strategy_name, path in history_map.items():
        if not path.exists():
            continue
        payload = load_json(path)
        val_acc = payload.get("val_acc", [])
        best_val = max(val_acc) if isinstance(val_acc, list) and val_acc else None
        out[strategy_name] = StrategyRecord(
            model_family=model_family,
            strategy=strategy_name,
            best_val_acc=clean_float(best_val),
            train_seconds=clean_float(payload.get("total_time")),
            trainable_params=clean_float(payload.get("trainable_params")),
            source_train=str(path),
        )
    return out


def load_efficient_eval(repo_root: Path) -> Dict[str, StrategyRecord]:
    eff_dir = repo_root / "EfficentNetV2"
    candidates = [
        eff_dir / "efficientnetv2_results_try2_with_inference.csv",
        eff_dir / "efficientnetv2_results_try2.csv",
    ]
    csv_path = next((p for p in candidates if p.exists()), None)
    if csv_path is None:
        return {}

    df = pd.read_csv(csv_path)
    out: Dict[str, StrategyRecord] = {}
    for _, row in df.iterrows():
        strategy = as_canonical_strategy(str(row.get("strategy", "")))
        if not strategy:
            continue
        out[strategy] = StrategyRecord(
            model_family="efficientnetv2",
            strategy=strategy,
            clean_test_acc=clean_float(row.get("clean_test_acc")),
            clean_test_loss=clean_float(row.get("clean_test_loss")),
            best_val_acc=clean_float(row.get("best_val_acc")),
            train_seconds=clean_float(row.get("train_seconds")),
            inference_ips=EFFICIENTNETV2_INFERENCE_IPS.get(strategy, clean_float(row.get("inference_ips"))),
            trainable_params=clean_float(row.get("trainable_params")),
            source_clean=str(csv_path),
            source_speed=str(csv_path),
            source_train=str(csv_path),
        )
    return out


def _iter_notebook_stream_texts(nb_path: Path) -> Iterable[str]:
    if not nb_path.exists():
        return []
    nb = load_json(nb_path)
    texts: List[str] = []
    for cell in nb.get("cells", []):
        for out in cell.get("outputs", []):
            if out.get("output_type") == "stream":
                text = out.get("text", [])
                if isinstance(text, list):
                    texts.append("".join(text))
                elif isinstance(text, str):
                    texts.append(text)
    return texts


def load_vit_eval(repo_root: Path) -> Dict[str, StrategyRecord]:
    vit_dir = repo_root / "Vision Transformer"
    csv_path = vit_dir / "vit_strategy_summary_all.csv"
    if csv_path.exists():
        df = pd.read_csv(csv_path)
        out: Dict[str, StrategyRecord] = {}
        for _, row in df.iterrows():
            strategy = as_canonical_strategy(str(row.get("strategy", "")))
            out[strategy] = StrategyRecord(
                model_family="vit",
                strategy=strategy,
                clean_test_acc=clean_float(row.get("clean_test_acc")),
                clean_test_loss=clean_float(row.get("clean_test_loss")),
                best_val_acc=clean_float(row.get("best_val_acc")),
                train_seconds=clean_float(row.get("train_seconds")),
                inference_ips=clean_float(row.get("inference_ips")),
                trainable_params=clean_float(row.get("trainable_params")),
                source_clean=str(csv_path),
                source_speed=str(csv_path),
                source_train=str(csv_path),
            )
        return out

    nb_path = vit_dir / "vit_analysis_try2.ipynb"
    streams = _iter_notebook_stream_texts(nb_path)
    summary_text = next(
        (
            s
            for s in streams
            if "Summary:" in s and "clean_test_acc" in s and "inference_ips" in s
        ),
        None,
    )
    if not summary_text:
        return {}

    part1: Dict[int, Tuple[str, float, float, float]] = {}
    part2: Dict[int, Tuple[Optional[float], Optional[float], Optional[float]]] = {}
    for line in summary_text.splitlines():
        m1 = re.match(
            r"^\s*(\d+)\s+([a-z_]+)\s+([0-9.]+)\s+([0-9.]+)\s+([0-9]+)\s*$",
            line,
        )
        if m1:
            idx = int(m1.group(1))
            part1[idx] = (
                as_canonical_strategy(m1.group(2)),
                float(m1.group(3)),
                float(m1.group(4)),
                float(m1.group(5)),
            )
            continue

        m2 = re.match(
            r"^\s*(\d+)\s+([0-9.]+|NaN|nan)\s+([0-9.]+)\s+([0-9.]+)\s*$",
            line,
        )
        if m2:
            idx = int(m2.group(1))
            part2[idx] = (
                clean_float(m2.group(2)),
                clean_float(m2.group(3)),
                clean_float(m2.group(4)),
            )

    out: Dict[str, StrategyRecord] = {}
    for idx, p1 in part1.items():
        strategy, clean_acc, clean_loss, trainable = p1
        p2 = part2.get(idx, (None, None, None))
        train_seconds, best_val_acc, inference_ips = p2
        out[strategy] = StrategyRecord(
            model_family="vit",
            strategy=strategy,
            clean_test_acc=clean_acc,
            clean_test_loss=clean_loss,
            best_val_acc=best_val_acc,
            train_seconds=train_seconds,
            inference_ips=inference_ips,
            trainable_params=trainable,
            source_clean=str(nb_path),
            source_speed=str(nb_path),
            source_train=str(nb_path),
        )
    return out


def load_resnet_eval(repo_root: Path) -> Dict[str, StrategyRecord]:
    res_dir = repo_root / "ResNet18"
    csv_path = res_dir / "resnet18_strategy_summary.csv"
    if csv_path.exists():
        df = pd.read_csv(csv_path)
        out: Dict[str, StrategyRecord] = {}
        for _, row in df.iterrows():
            strategy = as_canonical_strategy(str(row.get("strategy", "")))
            out[strategy] = StrategyRecord(
                model_family="resnet18",
                strategy=strategy,
                clean_test_acc=clean_float(row.get("clean_test_acc")),
                clean_test_loss=clean_float(row.get("clean_test_loss")),
                best_val_acc=clean_float(row.get("best_val_acc")),
                train_seconds=clean_float(row.get("train_seconds")),
                inference_ips=RESNET18_INFERENCE_IPS.get(strategy, clean_float(row.get("inference_ips"))),
                trainable_params=clean_float(row.get("trainable_params")),
                source_clean=str(csv_path),
                source_speed=str(csv_path),
                source_train=str(csv_path),
            )
        return out

    nb_path = res_dir / "resnet18.ipynb"
    streams = list(_iter_notebook_stream_texts(nb_path))
    if not streams:
        return {}

    test_acc: Dict[str, Tuple[float, float]] = {}
    infer_stats: Dict[str, Tuple[float, float]] = {}
    batch_size = 64.0

    test_line_re = re.compile(
        r"^\s*(Linear Probing|BatchNorm Tuning|Adapter Tuning|LoRA|Full Fine-tuning|Custom Model)\s+([0-9.]+)\s+([0-9.]+)\s*$"
    )
    infer_line_re = re.compile(
        r"^\s*(Linear Probing|BatchNorm Tuning|Adapter Tuning|LoRA|Full Fine-tuning|Custom Model)\s+\|\s+GFLOPs:\s+([0-9.]+)\s+\|\s+Infer time:\s+([0-9.]+)s\s*$"
    )

    for s in streams:
        for line in s.splitlines():
            m_test = test_line_re.match(line)
            if m_test:
                strat = as_canonical_strategy(m_test.group(1))
                test_acc[strat] = (float(m_test.group(2)), float(m_test.group(3)))
                continue

            m_infer = infer_line_re.match(line)
            if m_infer:
                strat = as_canonical_strategy(m_infer.group(1))
                gflops = float(m_infer.group(2))
                infer_time = float(m_infer.group(3))
                infer_ips = batch_size / infer_time if infer_time > 0 else None
                infer_stats[strat] = (gflops, infer_ips if infer_ips is not None else math.nan)

    out: Dict[str, StrategyRecord] = {}
    all_strats = set(test_acc) | set(infer_stats)
    for strat in all_strats:
        loss_acc = test_acc.get(strat, (math.nan, math.nan))
        _, acc = loss_acc
        gflops_ips = infer_stats.get(strat, (math.nan, math.nan))
        _, ips = gflops_ips
        out[strat] = StrategyRecord(
            model_family="resnet18",
            strategy=strat,
            clean_test_acc=clean_float(acc),
            clean_test_loss=clean_float(loss_acc[0]),
            inference_ips=RESNET18_INFERENCE_IPS.get(strat, clean_float(ips)),
            source_clean=str(nb_path),
            source_speed=str(nb_path),
        )
    return out


def merge_records(
    history_records: Dict[str, StrategyRecord],
    eval_records: Dict[str, StrategyRecord],
    model_family: str,
) -> List[StrategyRecord]:
    keys = sorted(set(history_records) | set(eval_records))
    rows: List[StrategyRecord] = []
    for k in keys:
        h = history_records.get(k)
        e = eval_records.get(k)
        rows.append(
            StrategyRecord(
                model_family=model_family,
                strategy=k,
                clean_test_acc=e.clean_test_acc if e else None,
                clean_test_loss=e.clean_test_loss if e else None,
                best_val_acc=(h.best_val_acc if h and h.best_val_acc is not None else (e.best_val_acc if e else None)),
                train_seconds=(h.train_seconds if h and h.train_seconds is not None else (e.train_seconds if e else None)),
                inference_ips=(EFFICIENTNETV2_INFERENCE_IPS.get(k, e.inference_ips if e else None) if model_family == "efficientnetv2" else (e.inference_ips if e else None)),
                trainable_params=(h.trainable_params if h and h.trainable_params is not None else (e.trainable_params if e else None)),
                source_clean=e.source_clean if e else "",
                source_speed=e.source_speed if e else "",
                source_train=(h.source_train if h and h.source_train else (e.source_train if e else "")),
            )
        )
    return rows


def records_to_df(records: List[StrategyRecord]) -> pd.DataFrame:
    return pd.DataFrame(
        [
            {
                "model_family": r.model_family,
                "strategy": r.strategy,
                "clean_test_acc": r.clean_test_acc,
                "clean_test_loss": r.clean_test_loss,
                "best_val_acc": r.best_val_acc,
                "train_seconds": r.train_seconds,
                "inference_ips": r.inference_ips,
                "trainable_params": r.trainable_params,
                "source_clean": r.source_clean,
                "source_speed": r.source_speed,
                "source_train": r.source_train,
            }
            for r in records
        ]
    )


def pick_best_by_model(df: pd.DataFrame) -> pd.DataFrame:
    def score_row(row: pd.Series) -> float:
        clean = row.get("clean_test_acc")
        best = row.get("best_val_acc")
        if pd.notna(clean):
            return float(clean)
        if pd.notna(best):
            return float(best)
        return float("-inf")

    rows = []
    for model, part in df.groupby("model_family", dropna=False):
        part = part.copy()
        part["__score__"] = part.apply(score_row, axis=1)
        best = part.sort_values("__score__", ascending=False).iloc[0].drop(labels="__score__")
        rows.append(best.to_dict())
    return pd.DataFrame(rows)


def scatter_plot(
    df: pd.DataFrame,
    x_col: str,
    y_col: str,
    title: str,
    x_label: str,
    y_label: str,
    out_path: Path,
) -> None:
    from matplotlib.lines import Line2D

    fig, ax = plt.subplots(figsize=(11, 7))
    family_colors = {
        "efficientnetv2": "#d97706",  # orange
        "resnet18": "#059669",        # green
        "vit": "#2563eb",             # blue
    }
    family_names = {
        "efficientnetv2": "EfficientNetV2",
        "resnet18": "ResNet18",
        "vit": "ViT",
    }
    strategy_names = {
        "linear_probe": "Linear Probe",
        "batchnorm_tuning": "BatchNorm",
        "adapter_tuning": "Adapter",
        "lora_tuning": "LoRA",
        "full_finetune": "Full Fine-Tuning",
        "custom_model": "Custom Model",
    }

    # Bubble size uses trainable parameters (log-scaled to keep sizes readable).
    valid_params = df["trainable_params"].dropna()
    if len(valid_params) > 0:
        log_min = math.log10(float(valid_params.min()))
        log_max = math.log10(float(valid_params.max()))
    else:
        log_min, log_max = 0.0, 1.0

    def bubble_size(param_val: Optional[float]) -> float:
        if param_val is None or pd.isna(param_val) or float(param_val) <= 0:
            return 280.0
        lp = math.log10(float(param_val))
        if abs(log_max - log_min) < 1e-9:
            return 500.0
        # point area in matplotlib scatter
        return 160.0 + (lp - log_min) / (log_max - log_min) * (1800.0 - 160.0)

    # Place labels around bubbles (not centered)
    offset_cycle = [(10, 8), (-10, 8), (10, -8), (-10, -8), (12, 0), (-12, 0), (0, 12), (0, -12)]

    for i, (_, row) in enumerate(df.iterrows()):
        x = row.get(x_col)
        y = row.get(y_col)
        if pd.isna(x) or pd.isna(y):
            continue

        family = row.get("model_family", "")
        strategy = row.get("strategy", "")
        color = family_colors.get(family, "#374151")
        size = bubble_size(row.get("trainable_params"))

        ax.scatter(
            x,
            y,
            s=size,
            color=color,
            alpha=0.55,
            edgecolors="black",
            linewidths=0.8,
            zorder=3,
        )

        label = strategy_names.get(strategy, str(strategy).replace("_", " ").title())
        dx, dy = offset_cycle[i % len(offset_cycle)]
        ha = "left" if dx > 0 else ("right" if dx < 0 else "center")
        va = "bottom" if dy > 0 else ("top" if dy < 0 else "center")
        ax.annotate(
            label,
            (x, y),
            textcoords="offset points",
            xytext=(dx, dy),
            fontsize=8,
            ha=ha,
            va=va,
            zorder=4,
        )

    # 3-color legend for model families.
    legend_handles = [
        Line2D([0], [0], marker="o", color="w", label=family_names[k], markerfacecolor=v, markeredgecolor="black", markersize=10)
        for k, v in family_colors.items()
    ]
    ax.legend(handles=legend_handles, title="Model Family", loc="best", frameon=True)

    ax.set_title(
        title + chr(10) + "(Bubble size = Trainable Paramaters)",
        pad=12,
    )
    ax.set_xlabel(x_label)
    ax.set_ylabel(y_label)
    ax.grid(alpha=0.25, zorder=1)
    fig.tight_layout()
    fig.savefig(out_path, dpi=220)
    plt.close(fig)


In [17]:
script_dir = Path.cwd() if Path.cwd().name == "Combined_Models" else Path.cwd() / "Combined_Models"
repo_root = script_dir.parent

# Histories
eff_hist = load_history_records(
    "efficientnetv2",
    {
        "linear_probe": repo_root / "EfficentNetV2/history_linear_efficientnetv2.json",
        "batchnorm_tuning": repo_root / "EfficentNetV2/history_bn_efficientnetv2.json",
        "adapter_tuning": repo_root / "EfficentNetV2/history_adapter_efficientnetv2.json",
        "lora_tuning": repo_root / "EfficentNetV2/history_lora_efficientnetv2.json",
        "full_finetune": repo_root / "EfficentNetV2/history_full_efficientnetv2.json",
    },
)
vit_hist = load_history_records(
    "vit",
    {
        "linear_probe": repo_root / "Vision Transformer/history_linear.json",
        "batchnorm_tuning": repo_root / "Vision Transformer/history_bn.json",
        "adapter_tuning": repo_root / "Vision Transformer/history_adapter.json",
        "lora_tuning": repo_root / "Vision Transformer/history_lora.json",
        "full_finetune": repo_root / "Vision Transformer/history_full.json",
    },
)
res_hist = load_history_records(
    "resnet18",
    {
        "linear_probe": repo_root / "ResNet18/history_s1_linear_probe.json",
        "batchnorm_tuning": repo_root / "ResNet18/history_s2_batchnorm.json",
        "adapter_tuning": repo_root / "ResNet18/history_s3_adapter.json",
        "lora_tuning": repo_root / "ResNet18/history_s4_lora.json",
        "full_finetune": repo_root / "ResNet18/history_s5_full_finetune.json",
        "custom_model": repo_root / "ResNet18/history_s6_custom.json",
    },
)

# Eval summaries
eff_eval = load_efficient_eval(repo_root)
vit_eval = load_vit_eval(repo_root)
res_eval = load_resnet_eval(repo_root)

all_records: List[StrategyRecord] = []
all_records.extend(merge_records(eff_hist, eff_eval, "efficientnetv2"))
all_records.extend(merge_records(vit_hist, vit_eval, "vit"))
all_records.extend(merge_records(res_hist, res_eval, "resnet18"))

df = records_to_df(all_records).sort_values(["model_family", "strategy"]).reset_index(drop=True)
out_csv = script_dir / "combined_strategies_tradeoff.csv"
df.to_csv(out_csv, index=False)

best_df = pick_best_by_model(df)
out_best_csv = script_dir / "combined_best_models_tradeoff.csv"
best_df.to_csv(out_best_csv, index=False)

# Plot-ready columns:
# - If clean test acc is missing (e.g., some EfficientNet full_finetune runs),
#   fallback to best validation accuracy so the point is still shown.
df_plot = df.copy()
df_plot["plot_perf"] = df_plot["clean_test_acc"]
mask_missing_perf = df_plot["plot_perf"].isna() & df_plot["best_val_acc"].notna()
df_plot.loc[mask_missing_perf, "plot_perf"] = df_plot.loc[mask_missing_perf, "best_val_acc"]

def infer_epoch_count_from_history(source_train: str) -> Optional[float]:
    if not source_train:
        return None
    p = Path(source_train)
    if not p.exists() or p.suffix.lower() != ".json":
        return None
    payload = load_json(p)
    for key in ("val_acc", "train_acc", "val_loss", "train_loss"):
        seq = payload.get(key)
        if isinstance(seq, list) and len(seq) > 0:
            return float(len(seq))
    return None

df_plot["epoch_count"] = df_plot["source_train"].apply(infer_epoch_count_from_history)
df_plot["epoch_time_seconds"] = pd.NA
valid_epoch_mask = (
    df_plot["train_seconds"].notna()
    & df_plot["epoch_count"].notna()
    & (df_plot["epoch_count"] > 0)
)
df_plot.loc[valid_epoch_mask, "epoch_time_seconds"] = (
    df_plot.loc[valid_epoch_mask, "train_seconds"] / df_plot.loc[valid_epoch_mask, "epoch_count"]
)
df_plot["epoch_time_seconds"] = pd.to_numeric(df_plot["epoch_time_seconds"], errors="coerce")
df_plot["plot_train_seconds"] = df_plot["train_seconds"]
df_plot["plot_epoch_time_seconds"] = df_plot["epoch_time_seconds"]
vit_mask = df_plot["model_family"] == "vit"
df_plot.loc[vit_mask, "plot_train_seconds"] = df_plot.loc[vit_mask, "plot_train_seconds"] * 2.0
df_plot.loc[vit_mask, "plot_epoch_time_seconds"] = df_plot.loc[vit_mask, "plot_epoch_time_seconds"] * 2.0

# If inference speed is missing, backfill for plotting with model-family median
# so strategies are still visible on the inference tradeoff chart.
df_plot["plot_inference_ips"] = df_plot["inference_ips"]
for fam, part in df_plot.groupby("model_family"):
    fam_med = part["plot_inference_ips"].median()
    if pd.notna(fam_med):
        fam_missing = (df_plot["model_family"] == fam) & df_plot["plot_inference_ips"].isna()
        df_plot.loc[fam_missing, "plot_inference_ips"] = fam_med

scatter_plot(
    df=df_plot,
    x_col="plot_train_seconds",
    y_col="plot_perf",
    title="Performance vs Wall-Clock Training Time",
    x_label="Training Time (seconds)",
    y_label="Performance (Clean Test Acc; fallback = Best Val Acc)",
    out_path=script_dir / "combined_tradeoff_wallclock.png",
)
scatter_plot(
    df=df_plot,
    x_col="plot_inference_ips",
    y_col="plot_perf",
    title="Performance vs Inference Speed",
    x_label="Inference Speed (images/sec)",
    y_label="Performance (Clean Test Acc; fallback = Best Val Acc)",
    out_path=script_dir / "combined_tradeoff_inference.png",
)
scatter_plot(
    df=df_plot,
    x_col="plot_epoch_time_seconds",
    y_col="plot_perf",
    title="Performance vs Epoch Time",
    x_label="Epoch Time (seconds/epoch)",
    y_label="Performance (Clean Test Acc; fallback = Best Val Acc)",
    out_path=script_dir / "combined_tradeoff_epoch_time.png",
)

print(f"Saved strategy-level table: {out_csv}")
print(f"Saved best-model table:     {out_best_csv}")
print(f"Saved plot:                 {script_dir / 'combined_tradeoff_wallclock.png'}")
print(f"Saved plot:                 {script_dir / 'combined_tradeoff_inference.png'}")
print(f"Saved plot:                 {script_dir / 'combined_tradeoff_epoch_time.png'}")

missing_clean = int(df["clean_test_acc"].isna().sum())
missing_speed = int(df["inference_ips"].isna().sum())
missing_epoch_time = int(df_plot["epoch_time_seconds"].isna().sum())
print(f"Missing clean_test_acc rows: {missing_clean}")
print(f"Missing inference_ips rows:  {missing_speed}")
print(f"Missing epoch_time rows:     {missing_epoch_time}")


Saved strategy-level table: /Users/jordankaseram/Documents/UBC/Data 586/Food-101/Combined_Models/combined_strategies_tradeoff.csv
Saved best-model table:     /Users/jordankaseram/Documents/UBC/Data 586/Food-101/Combined_Models/combined_best_models_tradeoff.csv
Saved plot:                 /Users/jordankaseram/Documents/UBC/Data 586/Food-101/Combined_Models/combined_tradeoff_wallclock.png
Saved plot:                 /Users/jordankaseram/Documents/UBC/Data 586/Food-101/Combined_Models/combined_tradeoff_inference.png
Saved plot:                 /Users/jordankaseram/Documents/UBC/Data 586/Food-101/Combined_Models/combined_tradeoff_epoch_time.png
Missing clean_test_acc rows: 2
Missing inference_ips rows:  0
Missing epoch_time rows:     0


In [18]:
print(df[['model_family','strategy','clean_test_acc','best_val_acc','train_seconds','inference_ips']].to_string(index=False))
print('\nBest per model family:')
print(best_df[['model_family','strategy','clean_test_acc','best_val_acc','train_seconds','inference_ips']].to_string(index=False))


  model_family         strategy  clean_test_acc  best_val_acc  train_seconds  inference_ips
efficientnetv2   adapter_tuning        0.761743      0.759208    1359.386785     404.531037
efficientnetv2 batchnorm_tuning        0.862099      0.825149    2895.296366     401.489927
efficientnetv2    full_finetune             NaN      0.840000    1367.648260     407.766329
efficientnetv2     linear_probe        0.646772      0.605545    2514.713800     403.022816
efficientnetv2      lora_tuning        0.841386      0.817822    1151.348015     349.559394
      resnet18   adapter_tuning        0.692200      0.648119     267.701058     343.449834
      resnet18 batchnorm_tuning        0.684700      0.640990     356.599941     300.089071
      resnet18     custom_model             NaN      0.723366     625.623136     333.118427
      resnet18    full_finetune        0.755300      0.710693     267.183059     333.118427
      resnet18     linear_probe        0.575500      0.532739     316.242637    